# Fireworks + Jev: one RL run from raw base

Reproduces the published recipe: the 24-update raw-base run behind the README and model card. It calls the same `fw-jev` CLI as the [tutorial](../docs/TUTORIAL.md): 96 training prompts, 24 evaluation prompts, at most 24 updates, one reward from start to finish. No SFT or restored checkpoint. Your run is a new stochastic run; it does not change the published results.

Run cells in order. Paid calls are disabled by default. Use a persistent CPU environment with network access to Fireworks, TypeSafe and Hugging Face. A local GPU is unnecessary. A disconnected/free notebook runtime is not reliable storage: back up `runs/`; never replay or resume uncertain optimizer calls.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = 'https://github.com/sophiamyang/fireworks_rl_jev_scorer.git'
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/fw_jev').is_dir()), Path.cwd() / 'fireworks_rl_jev_scorer')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
assert (REPO / 'experiments/raw-base-v1/config.json').exists(), 'Use the revision containing this notebook.'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.12.17'], check=True)
subprocess.run(['uv', 'sync', '--locked', '--python', '3.12'], cwd=REPO, check=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SMOKE = REPO / 'runs' / ('raw-smoke-' + RUN_ID)
TRAIN = REPO / 'runs' / ('raw-live-' + RUN_ID)
CONFIG = 'experiments/raw-base-v1/config.json'

def cli(*args):
    return subprocess.run(['uv', 'run', '--locked', 'python', '-m', 'fw_jev.cli', *map(str, args)], cwd=REPO, check=True)

## 1. Inspect the recipe

The reward is `(style + quality) / 2 × source_support`, with the invalid-output and high-confidence unsupported-claim guards. Source support is 1 for tasks such as fiction. No length penalty. The same Jev judge is used for training and evaluation; a higher score does not prove better writing or human authorship. This evaluation suite was previously inspected, not an untouched holdout.

In [ ]:
config = json.loads((REPO / CONFIG).read_text())
assert config['experiment'] == 'raw-simple-v1' and config['steps'] == 24
print(json.dumps(config, indent=2))
cli('plan', '--config', CONFIG)
print((REPO / 'experiments/raw-base-v1/contract.json').read_text())

## 2. Optional offline test

This exercises the trainer without model API calls. Mock output is not evidence of learning and cannot approve a live run.

In [ ]:
cli('run', '--config', CONFIG, '--output', REPO / 'runs' / ('raw-mock-' + RUN_ID), '--mock')

## 3. Enable paid calls and supply keys

Change `RUN_LIVE` only when ready. Keys stay in this kernel's environment; don't print them or save them in notebook outputs. Alternatively, the CLI can read a local ignored `.env`. W&B is off unless you set `USE_WANDB = True` (it then needs `WANDB_API_KEY`, and optionally `WANDB_ENTITY` for your team).

In [ ]:
RUN_LIVE = False
USE_WANDB = False
if RUN_LIVE:
    from getpass import getpass
    for name in ['FIREWORKS_API_KEY', 'TYPESAFE_API_KEY']:
        if not os.environ.get(name):
            os.environ[name] = getpass(name + ': ')
    if USE_WANDB and not os.environ.get('WANDB_API_KEY'):
        os.environ['WANDB_API_KEY'] = getpass('WANDB_API_KEY: ')
    os.environ.setdefault('WANDB_PROJECT', 'fireworks-rl-jev-scorer')
TRACKING = ['--wandb'] if USE_WANDB else []

## 4. Zero-update preflight

Scores 16 fixed drafts twice (32 Jev calls), then generates four drafts for each of six training prompts. No evaluation prompts and no optimizer updates. Stop if the fixed checks fail. Training re-checks this smoke automatically before it starts. Do not weaken the gate after viewing results.

In [ ]:
if RUN_LIVE:
    cli('run', '--config', 'experiments/raw-base-v1/smoke.json', '--output', SMOKE, '--execute', *TRACKING)
    print('Open:', SMOKE / 'comparison.html')
else:
    print('Paid preflight disabled.')

## 5. Look at the smoke drafts (optional review)

Open the smoke's `comparison.html` and read the drafts. Training checks the smoke automatically (fixed checks passed, valid drafts, at least three prompts with reward spread). For a stricter gate, fill in a full review and add `--require-review` to the training command; see the [recipe README](../experiments/raw-base-v1/README.md#optional-smoke-review).

In [ ]:
REQUIRE_REVIEW = False
if RUN_LIVE and REQUIRE_REVIEW:
    subprocess.run(['uv', 'run', '--locked', 'python', 'scripts/review_raw_smoke.py', 'init', str(SMOKE)], cwd=REPO, check=True)
    print('Edit after reading every draft:', SMOKE / 'smoke-review.json')
    print('Then check it: uv run python scripts/review_raw_smoke.py check', SMOKE)

## 6. Train once from raw base

This creates a new LoRA and optimizer, verifies model identity and uses the unchanged reward throughout. It does not load the smoke session or any saved checkpoint. Maximum: 768 training drafts + 96 before/after drafts, 896 Jev requests including repeated fixed checks, 24 optimizer updates. Fireworks costs are separate. Never rerun a failed output directory. Temperature is fixed at 1. The live run prints per-case rewards; if you watch them, record that exposure in the blind review.

In [ ]:
if RUN_LIVE:
    cli('run', '--config', CONFIG, '--smoke-from', SMOKE, '--output', TRAIN, '--execute', *TRACKING,
        *(['--require-review'] if REQUIRE_REVIEW else []))
    print('Trained. Do not open reports or HTML pages until the blind review is done.')
else:
    print('Paid training disabled.')

## 7. Blind review, then reveal

Generate the score-hidden A/B packet first: it contains only the packet, a blank ratings file and a random per-run `blind-key.json`, no scores. Read the packet and complete the ratings before opening the key, galleries, report or evaluation scores. If you already inspected those outputs or watched the live console, set `blinded_to_key_and_rewards` to false; the evaluation then reports the blind review as incomplete. Then run the final cell to reveal the evaluation and print the report. Inspect omissions, invented facts/actions, instruction leakage and lengths alongside reward.

In [ ]:
if RUN_LIVE:
    subprocess.run(['uv', 'run', '--locked', 'python', 'scripts/audit_run.py', str(TRAIN), '--output', str(TRAIN / 'local-audit.json')], cwd=REPO, check=True)
    subprocess.run(['uv', 'run', '--locked', 'python', 'experiments/scale-v1/evaluate.py', str(TRAIN), '--prepare-review'], cwd=REPO, check=True)
    print('Review packet:', TRAIN / 'blind-packet.json')
    print('Unapproved ratings:', TRAIN / 'blind-ratings.json')
    print('Read the packet and complete the ratings before continuing.')

In [ ]:
if RUN_LIVE:
    ratings = json.loads((TRAIN / 'blind-ratings.json').read_text())
    assert ratings.get('reviewer') and isinstance(ratings.get('blinded_to_key_and_rewards'), bool), 'Complete the review and record honestly whether you saw scores.'
    subprocess.run(['uv', 'run', '--locked', 'python', 'experiments/scale-v1/evaluate.py', str(TRAIN)], cwd=REPO, check=True)
    cli('report', TRAIN)
    print('Evaluation:', TRAIN / 'evaluation.json')
    print('All responses:', TRAIN / 'paired.html')
    print('Next: promote and download your adapter (tutorial step 6).')